In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.figsize"] = (20,10)

In [8]:
df = pd.read_csv("houserentdhaka_cleaned.csv")
df.head()

,Location,Area(sqft),Bed,Bath,Price(Taka)
0,"Block H, Bashundhara R-A, Dhaka",1600,3,3,20000
1,"Farmgate, Tejgaon, Dhaka",900,2,2,20000
2,"Block B, Nobodoy Housing Society, Mohammadpur,...",1250,3,3,18000
3,"Gulshan 1, Gulshan, Dhaka",2200,3,4,75000
4,"Baridhara, Dhaka",2200,3,3,75000


In [9]:
df.shape

(28800, 5)

## *Data Cleaning*

In [10]:
df.isna().sum()

Location       0
Area(sqft)     0
Bed            0
Bath           0
Price(Taka)    0
dtype: int64

In [11]:
df[df['Location']=='Dhaka']

,Location,Area(sqft),Bed,Bath,Price(Taka)
1195,Dhaka,700,2,1,10500
3213,Dhaka,1050,3,2,20000
9376,Dhaka,1100,3,3,18000
11494,Dhaka,1150,3,2,17500
16611,Dhaka,500,2,1,10000
16612,Dhaka,500,2,1,10000
16613,Dhaka,500,2,1,11000
16741,Dhaka,700,2,1,10500
16742,Dhaka,1486,3,3,20000
18280,Dhaka,1050,3,2,15500


In [12]:
df2 = df[df['Location']!='Dhaka']
df2.shape

(28787, 5)

## *Feature Engineering*

In [13]:
df2.head()

,Location,Area(sqft),Bed,Bath,Price(Taka)
0,"Block H, Bashundhara R-A, Dhaka",1600,3,3,20000
1,"Farmgate, Tejgaon, Dhaka",900,2,2,20000
2,"Block B, Nobodoy Housing Society, Mohammadpur,...",1250,3,3,18000
3,"Gulshan 1, Gulshan, Dhaka",2200,3,4,75000
4,"Baridhara, Dhaka",2200,3,3,75000


In [14]:
df2.Location.nunique()

729

In [16]:
def process_location(location):
    # Split by comma and strip whitespace
    parts = [part.strip() for part in location.split(",")]
    
    # Remove 'Dhaka' if it exists
    parts = [part for part in parts if part.lower() != "dhaka"]
    
    # Reverse the order
    parts.reverse()
    
    # Join the parts back into a single string
    return ", ".join(parts)

In [18]:
processed_locations = df2['Location'].apply(process_location)
processed_locations.head(10)

0                         Bashundhara R-A, Block H
1                                Tejgaon, Farmgate
2    Mohammadpur, Nobodoy Housing Society, Block B
3                               Gulshan, Gulshan 1
4                                        Baridhara
5                                  Bashundhara R-A
6                                        Baridhara
7                  Mohammadpur, PC Culture Housing
8                              Hazaribag, Jigatola
9                            Mirpur, West Kazipara
Name: Location, dtype: object

In [19]:
df3 = df2.copy()
df3['Location'] = processed_locations
df3.head()

,Location,Area(sqft),Bed,Bath,Price(Taka)
0,"Bashundhara R-A, Block H",1600,3,3,20000
1,"Tejgaon, Farmgate",900,2,2,20000
2,"Mohammadpur, Nobodoy Housing Society, Block B",1250,3,3,18000
3,"Gulshan, Gulshan 1",2200,3,4,75000
4,Baridhara,2200,3,3,75000


In [21]:
df4 = df3.copy()
df4['price_per_sqft'] = df4['Price(Taka)']/df4['Area(sqft)']
df4.head()

,Location,Area(sqft),Bed,Bath,Price(Taka),price_per_sqft
0,"Bashundhara R-A, Block H",1600,3,3,20000,12.500000
1,"Tejgaon, Farmgate",900,2,2,20000,22.222222
2,"Mohammadpur, Nobodoy Housing Society, Block B",1250,3,3,18000,14.400000
3,"Gulshan, Gulshan 1",2200,3,4,75000,34.090909
4,Baridhara,2200,3,3,75000,34.090909


In [23]:
df4.Location.nunique()

729

In [24]:
Location_stats = df4.groupby('Location')['Location'].agg('count').sort_values(ascending=False)
Location_stats.head(8)

Location
Mohammadpur                      757
Mirpur                           556
Mirpur, Section 12, Block D      417
Dhanmondi                        414
Mirpur, Section 12, Block E      411
Uttara, Sector 10                357
Mirpur, Ahmed Nagar, Paikpara    352
Mirpur, Kallyanpur               337
Name: Location, dtype: int64

In [28]:
less_than_5 = Location_stats[Location_stats<5]
len(less_than_5)

202

In [29]:
df4.Location = df4.Location.apply(lambda x: 'Other' if x in less_than_5 else x)
df4.Location.nunique()

528

In [30]:
df4[df4['Location']=='Other']

,Location,Area(sqft),Bed,Bath,Price(Taka),price_per_sqft
39,Other,800,3,2,15000,18.750000
69,Other,2420,3,4,45000,18.595041
135,Other,800,3,2,15000,18.750000
138,Other,1000,3,2,15000,15.000000
198,Other,1800,5,5,35000,19.444444
...,...,...,...,...,...,...
28541,Other,1100,3,2,12500,11.363636
28694,Other,740,2,2,16000,21.621622
28695,Other,780,2,2,16000,20.512821
28696,Other,740,2,2,16000,21.621622


In [31]:
df4[df4['Location']=='Other']['Price(Taka)'].mean()

np.float64(18108.773672055428)

In [32]:
df4.shape

(28787, 6)

### *Managing Outliers*

In [38]:
df4.price_per_sqft.describe()

count    28787.000000
mean        19.015292
std          6.553057
min          6.500000
25%         15.384615
50%         18.055556
75%         21.052632
max        228.571429
Name: price_per_sqft, dtype: float64

In [45]:
df4[df4.price_per_sqft>150]

,Location,Area(sqft),Bed,Bath,Price(Taka),price_per_sqft
26973,Shantinagar,2100,3,4,480000,228.571429


In [41]:
df4[df4.Location=='Shantinagar']['price_per_sqft'].mean()

np.float64(26.199116992090506)

In [49]:
df4[df4.price_per_sqft<7]

,Location,Area(sqft),Bed,Bath,Price(Taka),price_per_sqft
24748,Mirpur,2000,2,3,13000,6.5


In [72]:
#Removing rows using 3 sigma rule
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('Location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft>(m-3*st)) & (subdf.price_per_sqft<=(m+3*st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df5 = remove_pps_outliers(df4)
df5.shape

(28479, 6)

In [73]:
df5[df5.price_per_sqft>100]

,Location,Area(sqft),Bed,Bath,Price(Taka),price_per_sqft
4202,"Baridhara, Block K",3600,4,4,400000,111.111111
4204,"Baridhara, Block K",3600,4,4,400000,111.111111


In [74]:
df5[df5.price_per_sqft<7.5]

,Location,Area(sqft),Bed,Bath,Price(Taka),price_per_sqft
27763,"Uttara, Sector 18, Rajuk Uttara Apartment Proj...",1654,3,4,12000,7.255139
27764,"Uttara, Sector 18, Rajuk Uttara Apartment Proj...",1654,3,4,12000,7.255139
27769,"Uttara, Sector 18, Rajuk Uttara Apartment Proj...",1654,3,3,12000,7.255139
27779,"Uttara, Sector 18, Rajuk Uttara Apartment Proj...",1654,3,3,12000,7.255139
27783,"Uttara, Sector 18, Rajuk Uttara Apartment Proj...",1654,3,3,12000,7.255139
27787,"Uttara, Sector 18, Rajuk Uttara Apartment Proj...",1654,3,3,12000,7.255139
27856,"Uttara, Sector 18, Rajuk Uttara Apartment Proj...",1654,3,4,12000,7.255139
